In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")



In [25]:
df = pd.read_csv('../data/raw/data.csv', sep=';')
df.shape

(4424, 37)

In [26]:
print("Total columns:", len(df.columns))

for i, column in enumerate(df.columns, start=1):
    print(i, repr(column))

Total columns: 37
1 'Marital status'
2 'Application mode'
3 'Application order'
4 'Course'
5 'Daytime/evening attendance\t'
6 'Previous qualification'
7 'Previous qualification (grade)'
8 'Nacionality'
9 "Mother's qualification"
10 "Father's qualification"
11 "Mother's occupation"
12 "Father's occupation"
13 'Admission grade'
14 'Displaced'
15 'Educational special needs'
16 'Debtor'
17 'Tuition fees up to date'
18 'Gender'
19 'Scholarship holder'
20 'Age at enrollment'
21 'International'
22 'Curricular units 1st sem (credited)'
23 'Curricular units 1st sem (enrolled)'
24 'Curricular units 1st sem (evaluations)'
25 'Curricular units 1st sem (approved)'
26 'Curricular units 1st sem (grade)'
27 'Curricular units 1st sem (without evaluations)'
28 'Curricular units 2nd sem (credited)'
29 'Curricular units 2nd sem (enrolled)'
30 'Curricular units 2nd sem (evaluations)'
31 'Curricular units 2nd sem (approved)'
32 'Curricular units 2nd sem (grade)'
33 'Curricular units 2nd sem (without evaluat

In [27]:
print(df["Target"].value_counts())

print("\nTarget percentages:")

print(
    df["Target"].value_counts(normalize=True) * 100
)

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target percentages:
Target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64


In [28]:
# Remove leading/trailing spaces and tab characters
df.columns = df.columns.str.strip()

# Check the cleaned column names
for column in df.columns:
    print(repr(column))

'Marital status'
'Application mode'
'Application order'
'Course'
'Daytime/evening attendance'
'Previous qualification'
'Previous qualification (grade)'
'Nacionality'
"Mother's qualification"
"Father's qualification"
"Mother's occupation"
"Father's occupation"
'Admission grade'
'Displaced'
'Educational special needs'
'Debtor'
'Tuition fees up to date'
'Gender'
'Scholarship holder'
'Age at enrollment'
'International'
'Curricular units 1st sem (credited)'
'Curricular units 1st sem (enrolled)'
'Curricular units 1st sem (evaluations)'
'Curricular units 1st sem (approved)'
'Curricular units 1st sem (grade)'
'Curricular units 1st sem (without evaluations)'
'Curricular units 2nd sem (credited)'
'Curricular units 2nd sem (enrolled)'
'Curricular units 2nd sem (evaluations)'
'Curricular units 2nd sem (approved)'
'Curricular units 2nd sem (grade)'
'Curricular units 2nd sem (without evaluations)'
'Unemployment rate'
'Inflation rate'
'GDP'
'Target'


In [29]:
# Keep only students with resolved outcomes
binary_df = df[
    df["Target"].isin(["Dropout", "Graduate"])
].copy()

# Create binary target
binary_df["target"] = (
    binary_df["Target"] == "Dropout"
).astype(int)

# Check results
print("Original dataset:", df.shape)
print("Binary dataset:", binary_df.shape)

print("\nTarget distribution:")
print(binary_df["target"].value_counts())

print("\nTarget percentages:")
print(
    binary_df["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Original dataset: (4424, 37)
Binary dataset: (3630, 38)

Target distribution:
target
0    2209
1    1421
Name: count, dtype: int64

Target percentages:
target
0    60.85
1    39.15
Name: proportion, dtype: float64


In [30]:
missing = pd.DataFrame({
    "Missing Count": binary_df.isnull().sum(),
    "Missing Percentage": (
        binary_df.isnull().mean() * 100
    ).round(2)
})

missing = missing.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing)

,Missing Count,Missing Percentage
Marital status,0,0.0
Curricular units 2nd sem (enrolled),0,0.0
Curricular units 1st sem (credited),0,0.0
Curricular units 1st sem (enrolled),0,0.0
Curricular units 1st sem (evaluations),0,0.0
Curricular units 1st sem (approved),0,0.0
Curricular units 1st sem (grade),0,0.0
Curricular units 1st sem (without evaluations),0,0.0
Curricular units 2nd sem (credited),0,0.0
Curricular units 2nd sem (evaluations),0,0.0


In [31]:
print(
    "Duplicate rows:",
    binary_df.duplicated().sum()
)

print(
    "Unique student records:",
    binary_df.shape[0]
)

Duplicate rows: 0
Unique student records: 3630


In [32]:
print("Numerical columns:")
print(binary_df.select_dtypes(include=["number"]).columns.tolist())

print("\nCategorical columns:")
print(binary_df.select_dtypes(include=["object", "category"]).columns.tolist())

Numerical columns:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP', 'tar

In [33]:
semester_columns = [
    column for column in binary_df.columns
    if "1st sem" in column or "2nd sem" in column
]

print("Potential post-admission features:")

for column in semester_columns:
    print("-", column)

Potential post-admission features:
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 1st sem (approved)
- Curricular units 1st sem (grade)
- Curricular units 1st sem (without evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)


In [34]:
audit_summary = {
    "Original rows": len(df),
    "Original columns": len(df.columns),
    "Binary classification rows": len(binary_df),
    "Dropped unresolved outcomes": len(df) - len(binary_df),
    "Duplicate rows": binary_df.duplicated().sum(),
    "Missing cells": binary_df.isnull().sum().sum(),
    "Potential leakage columns": len(semester_columns)
}

for key, value in audit_summary.items():
    print(f"{key}: {value}")

Original rows: 4424
Original columns: 37
Binary classification rows: 3630
Dropped unresolved outcomes: 794
Duplicate rows: 0
Missing cells: 0
Potential leakage columns: 12



## Data Audit Findings

The selected dataset contains 4,424 student records
and 37 original columns.

The original target variable contains three outcomes:
Graduate, Dropout and Enrolled.

For the initial binary classification experiment,
students with an Enrolled outcome were excluded because
their final academic outcome was unresolved.

The resulting dataset contains 3,630 records:
2,209 graduates and 1,421 dropouts.

The dataset contains demographic information,
admission-related information and academic performance
information from two semesters.

Semester performance variables were identified as
potential leakage because they are recorded after
the admission prediction point.

These variables will be excluded from the initial
admission-time model.

Missing values: 0

Duplicate records: 0

The dataset is suitable for an initial supervised
classification experiment, subject to further
validation of feature definitions and data quality.
